# Breathing Numbers: Air Quality across Indian Cities, 2015–2020

An exploratory data analysis of CPCB's daily city-level air quality data.
Run top to bottom; each section poses a question, shows one chart, and leaves a cell for **your** reading of it.

> Prerequisite: `data/raw/city_day.csv` (see `data/README.md`).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display

from src.load_clean import load_clean
from src import analysis as an, plots, findings
from src.config import DEFAULT_DATA

FIG = ROOT / "figures"
df = load_clean(DEFAULT_DATA)
print(f"{len(df):,} rows | {df['City'].nunique()} cities | {df['Date'].min().date()} → {df['Date'].max().date()}")
df.head()

## 0. Know your data before you trust it

Missing values aren't uniformly distributed: cities joined the monitoring network at different times and stations go offline.
**Cleaning decision:** no imputation. Every analysis drops NaNs locally and states its own coverage threshold.

In [ ]:
display(df.isna().mean().sort_values(ascending=False).round(2).rename("share missing").to_frame().T)
display(Image(filename=str(plots.fig_coverage(df, FIG))))

**Observation:** _which cities have the longest/most complete records? Which are too sparse to compare?_

## 1. Which cities are the most polluted?

Bars show the *median* day; whiskers extend to the 95th-percentile day. A city can have a modest median and a terrible tail.

In [ ]:
display(Image(filename=str(plots.fig_city_ranking(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 2. How many days are actually 'Poor' or worse?

AQI averages hide how often people breathe genuinely unhealthy air. CPCB buckets: Good → Satisfactory → Moderate → Poor → Very Poor → Severe.

In [ ]:
display(Image(filename=str(plots.fig_bucket_shares(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 3. The Delhi story

Daily PM2.5 with a 30-day average, against India's 24-hour limit and the stricter WHO 2021 guideline.

In [ ]:
display(Image(filename=str(plots.fig_delhi_timeseries(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 4. Seasonality: the winter wall

Month × year heatmap of mean PM2.5. Look for the same band lighting up every year.

In [ ]:
display(Image(filename=str(plots.fig_delhi_calendar(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 5. Is the seasonal swing unique to the north?

Compare season distributions across northern, eastern, western and southern cities. Outliers hidden for readability.

In [ ]:
display(Image(filename=str(plots.fig_seasons(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 6. Does Diwali leave a fingerprint?

PM2.5 in the ±10 days around Diwali, one line per year. Caveat: weather and crop-residue burning also change over these weeks.

In [ ]:
display(Image(filename=str(plots.fig_diwali(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 7. A natural experiment: the 2020 lockdown

Mean pollutant levels during 25 Mar–31 May 2020 vs the *same dates in 2019*, which roughly controls for season. NO₂ tracks traffic closely; PM2.5 has many other sources.

In [ ]:
display(Image(filename=str(plots.fig_lockdown(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 8. Is it getting better?

Annual mean PM2.5 for selected cities. Only city-years with ≥240 days of data are plotted to avoid comparing a full year with a few weeks.

In [ ]:
display(Image(filename=str(plots.fig_trend(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## 9. What moves together?

Spearman correlations between pollutants, all cities pooled. Remember: pooling cities can create correlation (Simpson's paradox) that isn't there within a city.

In [ ]:
display(Image(filename=str(plots.fig_corr(df, FIG))))

**Observation:** _what do you see? What surprised you? What would you check next?_

## Headline numbers

The same numbers the README quotes, computed by code.

In [ ]:
f = findings.compute(df)
display(Markdown(findings.to_markdown(f)))

## Limitations & next steps

- City-level values aggregate stations (CPCB's own aggregation), hiding within-city variation.
- Missing data is not random; coverage differs by city and year.
- Comparisons (Diwali, lockdown) are associations: weather isn't controlled.
- **Next:** join daily weather (wind speed, temperature), add stubble-burning fire counts for the Delhi winter, or fit a simple seasonal model and test for a trend.